# Advanced Module 1 · Function Calling & Tools
Teaching the LLM to use tools — from a single function to parallel calls and routing.

---
> **Setup:** Run the first two cells before anything else.  
> API keys are loaded automatically from the `.env` file in the parent folder.

In [ ]:
%pip install -q google-genai groq python-dotenv

In [9]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types
from groq import Groq

load_dotenv()
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

groq_key = os.environ.get('Groq_key', '').strip()
if not groq_key:
    groq_key = getpass.getpass('Paste your Groq API key: ')

client      = genai.Client(api_key=api_key)
groq_client = Groq(api_key=groq_key)
MODEL       = 'gemini-3.1-flash-lite'
GROQ_MODEL  = 'llama-3.3-70b-versatile'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            elif '500' in msg or 'INTERNAL' in msg:
                wait = 10 * (attempt + 1)
                print(f'  ⏳ Server error — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen = client.models.generate_content
client.models.generate_content = lambda *a, **kw: _call_with_retry(_raw_gen, *a, **kw)

print('✅ Setup complete. Gemini:', MODEL, '| Groq:', GROQ_MODEL)

✅ Setup complete. Gemini: gemini-3.1-flash-lite | Groq: llama-3.3-70b-versatile


In [10]:
# API Key Test
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents='Reply with exactly the words: Hello LLM course',
        config=cfg(temperature=0.0)
    )
    print('✅ Gemini API working:', get_text(_r))
except Exception as e:
    print('❌ Gemini error:', e)

try:
    _rg = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'user', 'content': 'Reply with exactly: Hello Groq'}],
        max_tokens=10
    )
    print('✅ Groq API working:', _rg.choices[0].message.content)
except Exception as e:
    print('❌ Groq error:', e)

✅ Gemini API working: Hello LLM course
✅ Groq API working: Hello Groq


---
## The Core Idea: LLMs Don't Run Code — They Ask for It

A common misconception: "the AI called the weather API."  
The truth: **the LLM returned a structured JSON blob describing which function to call**. Your Python code ran it.

```
┌─────────────────────────────────────────────────────────────┐
│                  FUNCTION CALLING FLOW                      │
│                                                             │
│  User: "What's the stock price of AAPL?"                    │
│            │                                                │
│            ▼                                                │
│  ┌─────────────────┐                                        │
│  │   LLM decides:  │  ← receives tool definitions          │
│  │  "I need to     │                                        │
│  │   call a tool"  │                                        │
│  └────────┬────────┘                                        │
│           │                                                 │
│           ▼                                                 │
│  LLM returns:                                               │
│  { "function": "get_stock_price",                           │
│    "args": { "ticker": "AAPL" } }   ← NOT actual code!      │
│           │                                                 │
│           ▼                                                 │
│  ┌─────────────────┐                                        │
│  │  YOUR Python    │  ← you run get_stock_price("AAPL")     │
│  │  code executes  │     and get back "$189.50"             │
│  └────────┬────────┘                                        │
│           │                                                 │
│           ▼                                                 │
│  Send result back to LLM → LLM writes final answer          │
└─────────────────────────────────────────────────────────────┘
```

The LLM is the **decision-maker**. Your code is the **executor**.

---
## Demo 1: Single Tool — See the Raw Tool Call

Before executing anything, we print the raw `function_call` object the LLM returns.  
This is the moment students realize: **the LLM never touched Python**.

In [11]:
# Step 1: Define the tool as a Python function + JSON schema
import random

def get_stock_price(ticker: str) -> dict:
    """Stub: returns a fake stock price."""
    prices = {'AAPL': 189.50, 'GOOG': 175.30, 'MSFT': 415.20, 'AMZN': 198.75, 'TSLA': 248.60}
    price = prices.get(ticker.upper(), round(random.uniform(50, 500), 2))
    return {'ticker': ticker.upper(), 'price': price, 'currency': 'USD'}

# The schema tells the LLM what the tool does and what arguments it takes
stock_tool_schema = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name='get_stock_price',
            description='Returns the current stock price for a given ticker symbol.',
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    'ticker': types.Schema(
                        type=types.Type.STRING,
                        description='Stock ticker symbol, e.g. AAPL, GOOG'
                    )
                },
                required=['ticker']
            )
        )
    ]
)

print('✅ Tool defined:', stock_tool_schema.function_declarations[0].name)

✅ Tool defined: get_stock_price


In [12]:
# Step 2: Send the user question WITH the tool definition
user_question = "What is the current stock price of Apple?"

response = client.models.generate_content(
    model=MODEL,
    contents=user_question,
    config=cfg(tools=[stock_tool_schema], temperature=0.0)
)

# Step 3: Inspect what the LLM returned — BEFORE running any Python
print('=' * 60)
print('RAW RESPONSE FROM LLM (no Python executed yet!)')
print('=' * 60)
print(f'finish_reason : {response.candidates[0].finish_reason}')

# Search ALL parts for a function_call (it may not be in parts[0])
fc = next(
    (p.function_call for p in response.candidates[0].content.parts
     if hasattr(p, 'function_call') and p.function_call),
    None
)
if fc:
    print(f'\nfunction_call.name : {fc.name}')
    print(f'function_call.args : {dict(fc.args)}')
    print('\n👆 The LLM returned structured JSON — it never ran Python!')
else:
    print('No tool call made. LLM responded directly:', get_text(response))

RAW RESPONSE FROM LLM (no Python executed yet!)
finish_reason : FinishReason.STOP

function_call.name : get_stock_price
function_call.args : {'ticker': 'AAPL'}

👆 The LLM returned structured JSON — it never ran Python!


In [13]:
# Step 4: WE execute the function, then send the result back
# fc was set in the cell above — reuse it
if fc is None:
    print('⚠️  No tool call was made in the previous cell. Re-run that cell first.')
else:
    tool_result = get_stock_price(**dict(fc.args))
    print('Tool executed:', tool_result)

    # KEY: pass response.candidates[0].content directly — preserves thought_signature
    final_response = client.models.generate_content(
        model=MODEL,
        contents=[
            types.Content(role='user', parts=[types.Part(text=user_question)]),
            response.candidates[0].content,  # original model turn, not reconstructed
            types.Content(role='user', parts=[
                types.Part(function_response=types.FunctionResponse(
                    name=fc.name,
                    response={'result': tool_result}
                ))
            ])
        ],
        config=cfg(tools=[stock_tool_schema], temperature=0.0)
    )

    print('\n' + '=' * 60)
    print('FINAL ANSWER FROM LLM:')
    print('=' * 60)
    print(get_text(final_response))

Tool executed: {'ticker': 'AAPL', 'price': 189.5, 'currency': 'USD'}

FINAL ANSWER FROM LLM:
The current stock price of Apple (AAPL) is $189.50.


---
## Demo 2: Parallel Tool Calls — Ask for 3 Stocks at Once

When the LLM needs multiple independent pieces of data, it returns **multiple tool_call objects in one response**.  
You execute them in parallel, send all results back, and the LLM synthesizes everything.

In [14]:
user_question = "Compare the stock prices of Apple, Google, and Microsoft. Which is cheapest?"

response = client.models.generate_content(
    model=MODEL,
    contents=user_question,
    config=cfg(tools=[stock_tool_schema], temperature=0.0)
)

# Collect ALL function calls from the response
function_calls = [
    part.function_call
    for part in response.candidates[0].content.parts
    if hasattr(part, 'function_call') and part.function_call
]

print(f'LLM requested {len(function_calls)} tool calls in a single response:')
for fc in function_calls:
    print(f'  → {fc.name}({dict(fc.args)})')

# Execute all of them
tool_results = {}
for fc in function_calls:
    result = get_stock_price(**dict(fc.args))
    tool_results[fc.args['ticker']] = result
    print(f'  Executed: {result}')

# Build function response parts
result_parts = [
    types.Part(function_response=types.FunctionResponse(
        name=fc.name,
        response={'result': tool_results[fc.args['ticker']]}
    ))
    for fc in function_calls
]

final_response = client.models.generate_content(
    model=MODEL,
    contents=[
        types.Content(role='user', parts=[types.Part(text=user_question)]),
        response.candidates[0].content,  # original model turn — preserves thought_signature
        types.Content(role='user', parts=result_parts)
    ],
    config=cfg(tools=[stock_tool_schema], temperature=0.0)
)

print('\n' + '=' * 60)
print('FINAL ANSWER:')
print('=' * 60)
print(get_text(final_response))

LLM requested 3 tool calls in a single response:
  → get_stock_price({'ticker': 'AAPL'})
  → get_stock_price({'ticker': 'GOOG'})
  → get_stock_price({'ticker': 'MSFT'})
  Executed: {'ticker': 'AAPL', 'price': 189.5, 'currency': 'USD'}
  Executed: {'ticker': 'GOOG', 'price': 175.3, 'currency': 'USD'}
  Executed: {'ticker': 'MSFT', 'price': 415.2, 'currency': 'USD'}

FINAL ANSWER:
As of the latest data, here are the current stock prices for the three companies:

*   **Google (GOOG):** $175.30
*   **Apple (AAPL):** $189.50
*   **Microsoft (MSFT):** $415.20

Based on these prices, **Google (GOOG)** is the cheapest of the three.

***

*Disclaimer: Stock prices fluctuate constantly throughout the trading day. This information is for educational purposes and does not constitute financial advice.*


---
## Demo 3: Tool Chaining — Two Hops, One Conversation

The LLM first calls `search_product` to find a product, gets the price back,  
then calls `calculate_discount` to apply a discount — two sequential tool calls,  
each informed by the previous result.

In [15]:
# Define two tools
def search_product(query: str) -> dict:
    catalog = {
        'laptop': {'name': "ProBook 15'", 'price': 1299.99, 'product_id': 'PB15'},
        'headphones': {'name': 'SoundMax Pro', 'price': 249.99, 'product_id': 'SMP'},
        'phone': {'name': 'Galaxy Z6', 'price': 999.00, 'product_id': 'GZ6'},
    }
    for k, v in catalog.items():
        if k in query.lower():
            return v
    return {'name': 'Unknown Product', 'price': 0.0, 'product_id': 'UNK'}

def calculate_discount(original_price: float, discount_percent: float) -> dict:
    discount_amount = round(original_price * discount_percent / 100, 2)
    final_price     = round(original_price - discount_amount, 2)
    return {'original': original_price, 'discount': discount_amount, 'final': final_price}

multi_tool_schema = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name='search_product',
            description='Search the product catalog by keyword. Returns product name and price.',
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={'query': types.Schema(type=types.Type.STRING, description='Product search keyword')},
                required=['query']
            )
        ),
        types.FunctionDeclaration(
            name='calculate_discount',
            description='Calculate the discounted price given original price and discount percentage.',
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    'original_price':   types.Schema(type=types.Type.NUMBER, description='Original price in USD'),
                    'discount_percent': types.Schema(type=types.Type.NUMBER, description='Discount percentage (0-100)')
                },
                required=['original_price', 'discount_percent']
            )
        )
    ]
)

TOOL_MAP = {'search_product': search_product, 'calculate_discount': calculate_discount}

def run_tool(fc):
    fn = TOOL_MAP[fc.name]
    result = fn(**dict(fc.args))
    print(f'  🔧 Executed {fc.name}({dict(fc.args)}) → {result}')
    return result

# Agentic loop — keeps going until no more tool calls
user_question = "Find me a laptop and apply a 15% student discount. What's the final price?"
conversation  = [types.Content(role='user', parts=[types.Part(text=user_question)])]
step          = 0

print('User:', user_question)
print('-' * 60)

while True:
    step += 1
    response = client.models.generate_content(
        model=MODEL, contents=conversation,
        config=cfg(tools=[multi_tool_schema], temperature=0.0)
    )

    fcs = [
        part.function_call
        for part in response.candidates[0].content.parts
        if hasattr(part, 'function_call') and part.function_call
    ]

    if not fcs:  # No more tool calls → final answer
        print(f'\nStep {step}: Final answer (no more tools needed)')
        print('=' * 60)
        print(get_text(response))
        break

    print(f'Step {step}: LLM requests {len(fcs)} tool call(s):')
    results = [run_tool(fc) for fc in fcs]

    # KEY: use original content to preserve thought_signature
    conversation.append(response.candidates[0].content)
    conversation.append(types.Content(role='user', parts=[
        types.Part(function_response=types.FunctionResponse(name=fc.name, response={'result': r}))
        for fc, r in zip(fcs, results)
    ]))

User: Find me a laptop and apply a 15% student discount. What's the final price?
------------------------------------------------------------
Step 1: LLM requests 1 tool call(s):
  🔧 Executed search_product({'query': 'laptop'}) → {'name': "ProBook 15'", 'price': 1299.99, 'product_id': 'PB15'}
Step 2: LLM requests 1 tool call(s):
  🔧 Executed calculate_discount({'original_price': 1299.99, 'discount_percent': 15}) → {'original': 1299.99, 'discount': 195.0, 'final': 1104.99}

Step 3: Final answer (no more tools needed)
I found a ProBook 15' laptop for $1,299.99. After applying your 15% student discount, the final price is $1,104.99.


---
## Demo 4: Prompt Routing with Tools — The LLM as a Router

Instead of fixed `if/else` routing, we let the LLM decide which specialist handler to invoke  
based on the user's intent. The tool call **is** the routing decision.

In [16]:
def handle_technical(question: str) -> str:
    return f"[TECHNICAL HANDLER] Answering: {question}\nUse docs, be precise, include code examples."

def handle_emotional(message: str) -> str:
    return f"[EMPATHY HANDLER] Responding to: {message}\nBe warm, validate feelings, offer support."

def handle_math(problem: str) -> str:
    return f"[MATH HANDLER] Solving: {problem}\nShow step-by-step working."

router_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name='handle_technical',
            description='Route to technical specialist for coding, engineering, or factual questions.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={'question': types.Schema(type=types.Type.STRING)}, required=['question'])
        ),
        types.FunctionDeclaration(
            name='handle_emotional',
            description='Route to empathy specialist for emotional support, personal struggles, or mental health.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={'message': types.Schema(type=types.Type.STRING)}, required=['message'])
        ),
        types.FunctionDeclaration(
            name='handle_math',
            description='Route to math specialist for arithmetic, algebra, probability, or statistics.',
            parameters=types.Schema(type=types.Type.OBJECT,
                properties={'problem': types.Schema(type=types.Type.STRING)}, required=['problem'])
        )
    ]
)
ROUTER_MAP = {'handle_technical': handle_technical, 'handle_emotional': handle_emotional, 'handle_math': handle_math}

test_messages = [
    "How do I implement a binary search tree in Python?",
    "I've been feeling really anxious about my job lately and don't know what to do.",
    "What is the probability of rolling two sixes with two dice?"
]

for msg in test_messages:
    resp = client.models.generate_content(
        model=MODEL,
        contents=f'Route this user message to the appropriate handler: {msg}',
        config=cfg(tools=[router_tool], temperature=0.0)
    )
    fc = next(
        (p.function_call for p in resp.candidates[0].content.parts if hasattr(p, 'function_call') and p.function_call),
        None
    )
    if fc:
        handler_output = ROUTER_MAP[fc.name](**dict(fc.args))
        print(f'Input  : "{msg[:55]}..."' if len(msg) > 55 else f'Input  : "{msg}"')
        print(f'Routed → {fc.name}')
        print(f'Output : {handler_output[:80]}...')
        print()

Input  : "How do I implement a binary search tree in Python?"
Routed → handle_technical
Output : [TECHNICAL HANDLER] Answering: How do I implement a binary search tree in Python...

Input  : "I've been feeling really anxious about my job lately an..."
Routed → handle_emotional
Output : [EMPATHY HANDLER] Responding to: I've been feeling really anxious about my job l...

Input  : "What is the probability of rolling two sixes with two d..."
Routed → handle_math
Output : [MATH HANDLER] Solving: What is the probability of rolling two sixes with two di...



In [17]:
# ✏️  Define your own tool and wire it into the single-tool loop from Demo 1.
#    Ideas: get_weather(city), translate_text(text, target_lang), check_inventory(product_id)

def my_tool(input_text: str) -> dict:
    # TODO: implement your tool logic here
    return {'result': f'Processed: {input_text}'}

my_tool_schema = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name='my_tool',
            description='TODO: describe what your tool does',
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={'input_text': types.Schema(type=types.Type.STRING, description='Input text')},
                required=['input_text']
            )
        )
    ]
)

# Test it
my_question = 'TODO: write a question that would trigger your tool'
resp = client.models.generate_content(
    model=MODEL, contents=my_question,
    config=cfg(tools=[my_tool_schema], temperature=0.0)
)
fc = next(
    (p.function_call for p in resp.candidates[0].content.parts if hasattr(p, 'function_call') and p.function_call),
    None
)
if fc:
    print('Tool called:', fc.name, dict(fc.args))
    print('Result:', my_tool(**dict(fc.args)))
else:
    print('No tool call — try rephrasing your question or improving the tool description.')

Tool called: my_tool {'input_text': 'What is the capital of France?'}
Result: {'result': 'Processed: What is the capital of France?'}


---
## Key Takeaways

| Concept | What it means |
|---|---|
| Function calling | LLM returns a JSON blob; your code runs the actual function |
| Tool schema | JSON description of function name, args, and types — the LLM's API contract |
| Parallel tool calls | LLM can request multiple tools in one response for independent data |
| Tool chaining | Multi-hop: result of tool A becomes input to tool B in the same conversation |
| Routing with tools | LLM routes to specialist handlers by choosing which tool to call |
| Gemini vs Groq | Same tool-use pattern; Groq uses OpenAI-compatible JSON; Gemini uses `types.Tool` |